In [0]:
import os
import psutil

proc = psutil.Process(os.getpid())
vm = psutil.virtual_memory()

print("Process RAM (GB):", round(proc.memory_info().rss / 1024**3, 3))
print("System used (GB):", round(vm.used / 1024**3, 3))
print("System total (GB):", round(vm.total / 1024**3, 3))
print("System percent:", vm.percent)

Process RAM (GB): 0.361
System used (GB): 8.673
System total (GB): 30.729
System percent: 29.2


In [0]:
VOLUME = "/Volumes/workspace/default/airline_dataset"
CSV_NAME = "airline.csv.shuffle"

DATA_PATH = f"{VOLUME}/{CSV_NAME}"
OUTPUT_DIR = f"{VOLUME}/output"

BENCHMARK_GB = 1.0

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Data  :", DATA_PATH)
print("Output:", OUTPUT_DIR)
print("Exists:", os.path.exists(DATA_PATH))

Data  : /Volumes/workspace/default/airline_dataset/airline.csv.shuffle
Output: /Volumes/workspace/default/airline_dataset/output
Exists: True


In [0]:
import csv
import time
from contextlib import contextmanager

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("AirlineDelayAnalysis").getOrCreate()

STAGE_TIMES = {}

@contextmanager
def time_stage(name):
    box = {"rows": None}
    t0 = time.perf_counter()
    yield box
    wall = round(time.perf_counter() - t0, 3)
    STAGE_TIMES[name] = {"Rows Processed": box["rows"], "Wall Time (s)": wall}
    print(f"[{wall:9.3f}s] {name}")

def show(df, n=30):
    try:
        display(df)
    except NameError:
        df.show(n, truncate=False)

DELAY_CAUSES = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
USECOLS = [
    "Year", "Month", "DayofMonth", "DayOfWeek",
    "UniqueCarrier", "Origin", "Dest", "Distance",
    "DepDelay", "ArrDelay", "Cancelled", "Diverted", "CancellationCode",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]

In [0]:
with time_stage("0. Disk read benchmark (raw bytes)") as t:
    limit = int(BENCHMARK_GB * 1024 ** 3)
    read_bytes = 0
    with open(DATA_PATH, "rb") as f:
        while read_bytes < limit:
            chunk = f.read(64 * 1024 * 1024)
            if not chunk:
                break
            read_bytes += len(chunk)

print(f"{read_bytes / 1024 ** 3:.2f} GB read")

[    2.902s] 0. Disk read benchmark (raw bytes)
1.00 GB read


In [0]:
with time_stage("1. Data loading (read_csv, chunked)") as t:
    raw = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("nullValue", "NA")
        .csv(DATA_PATH)
    )
    raw_count = raw.count()
    t["rows"] = raw_count

print(f"{raw_count:,} rows loaded")

[  129.117s] 1. Data loading (read_csv, chunked)
123,534,969 rows loaded


In [0]:
with time_stage("2a. Data quality audit (pre-clean)") as t:
    pruned = raw.select(USECOLS)
    null_counts = pruned.agg(
        *[F.sum(F.col(c).isNull().cast("long")).alias(c) for c in USECOLS]
    ).collect()[0].asDict()
    t["rows"] = raw_count

audit = pd.DataFrame(
    [{"Column": c, "Nulls": n, "Null %": round(100 * n / raw_count, 3)} for c, n in null_counts.items()]
).sort_values("Nulls", ascending=False)
print(audit.to_string(index=False))

[   46.379s] 2a. Data quality audit (pre-clean)
           Column     Nulls  Null %
 CancellationCode 122800263  99.405
LateAircraftDelay  89329433  72.311
    SecurityDelay  89329433  72.311
         NASDelay  89329433  72.311
     WeatherDelay  89329433  72.311
     CarrierDelay  89329433  72.311
         ArrDelay   2587529   2.095
         DepDelay   2302136   1.864
         Distance    202000   0.164
            Month         0   0.000
        Cancelled         0   0.000
         Diverted         0   0.000
             Dest         0   0.000
           Origin         0   0.000
    UniqueCarrier         0   0.000
        DayOfWeek         0   0.000
       DayofMonth         0   0.000
             Year         0   0.000


In [0]:
with time_stage("2b. Cleaning (filter + downcast)") as t:
    c1 = F.col("Year").between(1987, 2030)
    c2 = c1 & F.col("Month").between(1, 12)
    c3 = c2 & (F.col("Cancelled") == 0)
    c4 = c3 & (F.col("Diverted") == 0)
    c5 = c4 & F.col("ArrDelay").isNotNull()
    c6 = c5 & (F.abs(F.col("ArrDelay")) <= 1440) & \
         (F.abs(F.coalesce(F.col("DepDelay"), F.lit(0))) <= 1440)

    counts = pruned.agg(
        F.sum(F.when(c1, 1).otherwise(0)).alias("c1"),
        F.sum(F.when(c2, 1).otherwise(0)).alias("c2"),
        F.sum(F.when(c3, 1).otherwise(0)).alias("c3"),
        F.sum(F.when(c4, 1).otherwise(0)).alias("c4"),
        F.sum(F.when(c5, 1).otherwise(0)).alias("c5"),
        F.sum(F.when(c6, 1).otherwise(0)).alias("c6"),
    ).collect()[0]

    clean_log = []
    prev = raw_count
    for name, key in zip(
        ["1. Valid Year", "2. Valid Month (1-12)", "3. Not cancelled",
         "4. Not diverted", "5. ArrDelay present", "6. |delay| <= 1440 min"],
        ["c1", "c2", "c3", "c4", "c5", "c6"],
    ):
        cur = counts[key] or 0
        clean_log.append({"Rule": name, "Rows removed": prev - cur, "Rows remaining": cur})
        prev = cur

    df = pruned.filter(c6).withColumn("Delayed15", F.col("ArrDelay") > 15).dropDuplicates()
    df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/clean_parquet")
    df = spark.read.parquet(f"{OUTPUT_DIR}/clean_parquet")
    clean_count = df.count()
    clean_log.append({
        "Rule": "7. Drop exact duplicates",
        "Rows removed": counts["c6"] - clean_count,
        "Rows remaining": clean_count,
    })
    t["rows"] = clean_count

print(pd.DataFrame(clean_log).to_string(index=False))

with open(f"{OUTPUT_DIR}/clean_log_spark.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["Rule", "Rows removed", "Rows remaining"])
    w.writeheader()
    w.writerows(clean_log)

[  156.532s] 2b. Cleaning (filter + downcast)
                    Rule  Rows removed  Rows remaining
           1. Valid Year             0       123534969
   2. Valid Month (1-12)             0       123534969
        3. Not cancelled       2303324       121231645
         4. Not diverted        284204       120947441
     5. ArrDelay present             1       120947440
  6. |delay| <= 1440 min            83       120947357
7. Drop exact duplicates       2444026       118503331


In [0]:
with time_stage("3a. Schema description") as t:
    schema = [(f.name, f.dataType.simpleString()) for f in df.schema.fields]
    t["rows"] = clean_count

print(pd.DataFrame(schema, columns=["Column", "Type"]).to_string(index=False))

[    0.000s] 3a. Schema description
           Column    Type
             Year     int
            Month     int
       DayofMonth     int
        DayOfWeek     int
    UniqueCarrier  string
           Origin  string
             Dest  string
         Distance     int
         DepDelay     int
         ArrDelay     int
        Cancelled     int
         Diverted     int
 CancellationCode  string
     CarrierDelay     int
     WeatherDelay     int
         NASDelay     int
    SecurityDelay     int
LateAircraftDelay     int
        Delayed15 boolean


In [0]:
NUMERIC = ["Year", "Month", "DayofMonth", "DayOfWeek", "Distance", "DepDelay", "ArrDelay"] + DELAY_CAUSES

with time_stage("3b. Summary statistics (describe)") as t:
    describe_pd = df.select(NUMERIC).describe().toPandas()
    t["rows"] = clean_count

show(describe_pd)

[    4.510s] 3b. Summary statistics (describe)


summary,Year,Month,DayofMonth,DayOfWeek,Distance,DepDelay,ArrDelay,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
count,118503331,118503331,118503331,118503331,118309103,118503331,118503331,33063650,33063650,33063650,33063650,33063650
mean,1998.6334876021333,6.563563559238685,15.733695840161658,3.9496612377925477,708.8446298337669,8.283010778827812,7.258004857264307,3.8066753368124813,0.815475544895981,4.243575709275897,0.027623810438351482,4.919964039057999
stddev,6.243694364587928,3.439252153398546,8.791196036431444,1.9909081477546018,554.2035188701443,28.517641721642686,30.986993702881797,20.08612927869664,9.590853781905324,16.857712659928303,1.2140839587245384,20.53935469771703
min,1987,1,1,1,0,-1410,-1437,0,0,-60,0,0
max,2008,12,31,7,4983,1439,1438,1431,1429,1392,533,1366


In [0]:
CATEGORICAL = ["UniqueCarrier", "Origin", "Dest", "CancellationCode"]

with time_stage("3c. Categorical description") as t:
    distinct = df.agg(
        *[F.countDistinct(F.col(c)).alias(c) for c in CATEGORICAL]
    ).collect()[0].asDict()
    t["rows"] = clean_count

print(pd.DataFrame(
    [{"Column": c, "Distinct values": n} for c, n in distinct.items()]
).to_string(index=False))

[    2.024s] 3c. Categorical description
          Column  Distinct values
   UniqueCarrier               29
          Origin              347
            Dest              343
CancellationCode                3


In [0]:
HIST_BIN_W, HIST_LO, HIST_HI = 5, -60, 180

with time_stage("4. EDA 1 - delay distribution (histogram binning)") as t:
    binned = df.withColumn("bin", F.floor((F.col("ArrDelay") - HIST_LO) / HIST_BIN_W))
    hist_sdf = (
        binned.filter((F.col("ArrDelay") >= HIST_LO) & (F.col("ArrDelay") <= HIST_HI))
        .groupBy("bin").count()
        .withColumn("bin_left", F.col("bin") * HIST_BIN_W + HIST_LO)
        .orderBy("bin")
    )
    n_below = df.filter(F.col("ArrDelay") < HIST_LO).count()
    n_above = df.filter(F.col("ArrDelay") > HIST_HI).count()
    percentiles = df.approxQuantile("ArrDelay", [0.01, 0.05, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99, 0.999], 0.001)
    mean_delay = df.select(F.avg("ArrDelay")).collect()[0][0]
    t["rows"] = clean_count

print(f"Mean {mean_delay:.3f} min, median {percentiles[3]:.1f} min")
print(f"Outside [{HIST_LO}, {HIST_HI}]: {n_below:,} below, {n_above:,} above")
show(hist_sdf.select("bin_left", "count"))

[    3.141s] 4. EDA 1 - delay distribution (histogram binning)
Mean 7.258 min, median 0.0 min
Outside [-60, 180]: 3,681 below, 553,029 above


bin_left,count
-60,3982
-55,9473
-50,23415
-45,58891
-40,149671
-35,378228
-30,950108
-25,2302061
-20,5226842
-15,10410520


In [0]:
with time_stage("5. EDA 2 - airline comparison (groupby aggregation)") as t:
    by_carrier_sdf = df.groupBy("UniqueCarrier").agg(
        F.count("*").alias("flights"),
        F.avg("ArrDelay").alias("mean_delay"),
        F.expr("percentile_approx(ArrDelay, 0.5)").alias("median_delay"),
        (F.avg(F.when(F.col("Delayed15"), 1.0).otherwise(0.0)) * 100).alias("pct_delayed_15"),
    )
    carrier_pd = by_carrier_sdf.toPandas().set_index("UniqueCarrier")
    t["rows"] = clean_count

ranked = carrier_pd[carrier_pd["flights"] >= max(1000, int(0.0005 * clean_count))].sort_values("mean_delay")
print(ranked.round(2))
show(by_carrier_sdf)

[    3.840s] 5. EDA 2 - airline comparison (groupby aggregation)
                flights  mean_delay  median_delay  pct_delayed_15
UniqueCarrier                                                    
HA               258486       -0.56            -4            6.46
AQ               145318        1.31            -2            9.05
ML (1)            64177        5.35             0           14.42
NW              9982197        5.54            -1           18.18
F9               333680        5.73             0           18.85
PA (1)           294645        5.91             0           19.40
OO              2942603        6.11            -2           17.57
9E               503028        6.14            -4           18.77
TZ               205428        6.17            -3           19.05
WN             14453463        6.32             0           17.73
US             13664382        6.51             0           19.13
DH               666849        6.87            -4           21.26
AA         

UniqueCarrier,flights,mean_delay,median_delay,pct_delayed_15
TZ,205428,6.166014369998248,-3,19.045602352162312
PA (1),294645,5.909894958339697,0,19.397919530282202
F9,333680,5.729905897866219,0,18.854890913450014
ML (1),64177,5.3495489038128925,0,14.424170653037693
PI,857344,10.517876138399522,5,23.48508883248731
HA,258486,-0.5563473456976393,-4,6.45992432859033
AS,2740537,8.60657528068404,2,21.589819805388505
EA,841937,7.609112083208126,0,18.50946092166041
EV,1635921,10.839068634732362,0,25.59909677790064
CO,7940281,7.0594273678727495,0,19.571574356121655


In [0]:
with time_stage("6. EDA 3 - monthly trend (groupby aggregation)") as t:
    by_month_sdf = (
        df.groupBy("Month")
        .agg(
            F.count("*").alias("flights"),
            F.avg("ArrDelay").alias("mean_delay"),
            F.expr("percentile_approx(ArrDelay, 0.5)").alias("median_delay"),
            (F.avg(F.when(F.col("Delayed15"), 1.0).otherwise(0.0)) * 100).alias("pct_delayed_15"),
        )
        .orderBy("Month")
    )
    by_month_pd = by_month_sdf.toPandas()
    year_min, year_max = df.select(F.min("Year"), F.max("Year")).collect()[0]
    t["rows"] = clean_count

by_month_pd["MonthName"] = by_month_pd["Month"].apply(lambda m: MONTH_NAMES[int(m) - 1])
print(by_month_pd.set_index("MonthName")[["flights", "mean_delay", "median_delay", "pct_delayed_15"]].round(2))
print(f"{clean_count:,} flights across {year_min}-{year_max}")
show(by_month_sdf)

[    3.814s] 6. EDA 3 - monthly trend (groupby aggregation)
            flights  mean_delay  median_delay  pct_delayed_15
MonthName                                                    
Jan         9718721        8.67             1           22.44
Feb         8974597        8.11             1           21.54
Mar        10007561        7.45             0           20.40
Apr         9725754        5.44             0           17.25
May         9966350        5.68            -1           17.28
Jun         9841218        9.95             1           22.12
Jul        10174710        9.08             0           20.90
Aug        10250222        7.97             0           20.04
Sep         9464386        3.58            -2           14.77
Oct        10378287        4.93             0           16.62
Nov         9873917        5.46             0           18.05
Dec        10127608       10.68             2           25.13
118,503,331 flights across 1987-2008


Month,flights,mean_delay,median_delay,pct_delayed_15
1,9718721,8.665095437969667,1,22.43744830209654
2,8974597,8.111030166591325,1,21.544889425118477
3,10007561,7.451202945452943,0,20.395179205003096
4,9725754,5.437580777798821,0,17.2541069823481
5,9966350,5.676905386625997,-1,17.279545671183534
6,9841218,9.945180057996886,1,22.11679489266471
7,10174710,9.07887723581311,0,20.897804458308887
8,10250222,7.974221533933607,0,20.035566058959503
9,9464386,3.5833083096991185,-2,14.769399726511578
10,10378287,4.9259229389204595,0,16.620228367166952


In [0]:
with time_stage("7. EDA 4 - delay cause breakdown") as t:
    cause_exprs = [F.sum(F.when(F.col(c) > 0, F.col(c)).otherwise(0)).alias(c) for c in DELAY_CAUSES]
    cause_totals = df.agg(*cause_exprs).collect()[0].asDict()
    t["rows"] = clean_count

total_min = sum(cause_totals.values()) or 1
for cause, minutes in sorted(cause_totals.items(), key=lambda kv: -kv[1]):
    print(f"{cause:<20}{minutes:>14,.1f} min  ({100 * minutes / total_min:5.1f}%)")

cause_df = spark.createDataFrame(
    [(c, float(m), 100 * m / total_min) for c, m in cause_totals.items()],
    ["Cause", "Minutes", "Pct"],
).orderBy(F.desc("Minutes"))
show(cause_df)

[    1.053s] 7. EDA 4 - delay cause breakdown
LateAircraftDelay    162,671,969.0 min  ( 35.6%)
NASDelay             140,309,065.0 min  ( 30.7%)
CarrierDelay         125,862,581.0 min  ( 27.6%)
WeatherDelay          26,962,598.0 min  (  5.9%)
SecurityDelay            913,344.0 min  (  0.2%)


Cause,Minutes,Pct
LateAircraftDelay,1.62671969E8,35.617473897663636
NASDelay,1.40309065E8,30.721054714983445
CarrierDelay,1.25862581E8,27.55795740973711
WeatherDelay,2.6962598E7,5.903534803087051
SecurityDelay,913344.0,0.1999791745287579


In [0]:
by_carrier_sdf.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OUTPUT_DIR}/carrier_summary")
by_month_sdf.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OUTPUT_DIR}/trend_summary")
hist_sdf.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OUTPUT_DIR}/histogram_summary")
print("Written to", OUTPUT_DIR)

Written to /Volumes/workspace/default/airline_dataset/output


In [0]:
metrics = pd.DataFrame(
    [{"Stage": k, "Rows Processed": v["Rows Processed"], "Wall Time (s)": v["Wall Time (s)"]}
     for k, v in STAGE_TIMES.items()]
)
metrics.loc[len(metrics)] = {
    "Stage": "TOTAL",
    "Rows Processed": clean_count,
    "Wall Time (s)": round(metrics["Wall Time (s)"].sum(), 3),
}

metrics.to_csv(f"{OUTPUT_DIR}/spark_stage_times.csv", index=False)
show(metrics)
print(metrics.to_string(index=False))

Stage,Rows Processed,Wall Time (s)
0. Disk read benchmark (raw bytes),null,2.902
"1. Data loading (read_csv, chunked)",1.23534969E8,129.117
2a. Data quality audit (pre-clean),1.23534969E8,46.379
2b. Cleaning (filter + downcast),1.18503331E8,156.532
3a. Schema description,1.18503331E8,0.0
3b. Summary statistics (describe),1.18503331E8,4.51
3c. Categorical description,1.18503331E8,2.024
4. EDA 1 - delay distribution (histogram binning),1.18503331E8,3.141
5. EDA 2 - airline comparison (groupby aggregation),1.18503331E8,3.84
6. EDA 3 - monthly trend (groupby aggregation),1.18503331E8,3.814


                                              Stage  Rows Processed  Wall Time (s)
                 0. Disk read benchmark (raw bytes)             NaN          2.902
                1. Data loading (read_csv, chunked)     123534969.0        129.117
                 2a. Data quality audit (pre-clean)     123534969.0         46.379
                   2b. Cleaning (filter + downcast)     118503331.0        156.532
                             3a. Schema description     118503331.0          0.000
                  3b. Summary statistics (describe)     118503331.0          4.510
                        3c. Categorical description     118503331.0          2.024
  4. EDA 1 - delay distribution (histogram binning)     118503331.0          3.141
5. EDA 2 - airline comparison (groupby aggregation)     118503331.0          3.840
     6. EDA 3 - monthly trend (groupby aggregation)     118503331.0          3.814
                   7. EDA 4 - delay cause breakdown     118503331.0          1.053
    